# 03 - Entrenamiento YOLO11 para detección de humo y fuego (standalone)

## Salidas esperadas

- pesos del modelo (`best.pt` y `last.pt`) en Drive;
- métricas de entrenamiento y validación;
- curvas de desempeño;
- matriz de confusión;
- carpeta de resultados asociada al experimento, en Drive.

In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import time
import yaml
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. Para entrenar se recomienda activar GPU en Colab.")

In [ ]:
# ============================================================
# Montar Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta principal en Drive:", DRIVE_PROJECT_DIR)
print("Carpeta de corridas:", DRIVE_RUNS_DIR)

In [ ]:
# ============================================================
# Instalación de dependencias (directo, sin tocar el repo)
# ============================================================
# Solo se fuerza el upgrade de ultralytics (necesario para tener soporte
# de YOLO11 al día). El resto se instala sin -U para no pisar las
# versiones de numpy/pandas/torch que ya trae Colab preinstaladas y
# de las que dependen paquetes como google-colab y numba.

!pip install -q -U ultralytics
!pip install -q opencv-python-headless matplotlib pandas pyyaml kagglehub numpy torch torchvision tqdm

print("Dependencias instaladas.")

In [ ]:
# ============================================================
# Descarga y localización del dataset D-Fire (Kaggle)
# ============================================================

import kagglehub

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"

dataset_root = Path(kagglehub.dataset_download(DATASET_ID))

print("Dataset descargado/localizado en:")
print(dataset_root)


def find_yolo_dataset_dir(root: Path) -> Path:
    """
    Busca automáticamente la carpeta que contiene la estructura esperada:
    train/images, train/labels, val/images, val/labels.
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for candidate in candidates:
        required = [
            candidate / "train" / "images",
            candidate / "train" / "labels",
            candidate / "val" / "images",
            candidate / "val" / "labels",
        ]

        if all(path.exists() for path in required):
            return candidate

    raise FileNotFoundError(
        "No se encontró una estructura YOLO válida con train/images, train/labels, val/images y val/labels."
    )


DATA_DIR = find_yolo_dataset_dir(dataset_root)

print("Carpeta de datos YOLO detectada:")
print(DATA_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split
    print(f"{split}: existe={split_dir.exists()} -> {split_dir}")

In [ ]:
# ============================================================
# Generación del YAML de dataset para YOLO
# ============================================================

DFIRE_YAML = Path("/content/dfire_colab.yaml")

dfire_config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with open(DFIRE_YAML, "w", encoding="utf-8") as file:
    yaml.safe_dump(dfire_config, file, sort_keys=False, allow_unicode=True)

print("Archivo YAML de dataset creado en:")
print(DFIRE_YAML)

print("\nContenido:")
with open(DFIRE_YAML, "r", encoding="utf-8") as file:
    print(file.read())

In [ ]:
# ============================================================
# Config del experimento (equivalente a
# configs/experiments/yolo11n_baseline.yaml, definida inline)
# ============================================================

experiment_config = {
    "experiment": {
        "name": "yolo11n_baseline",
        "family": "YOLO11",
        "model": "yolo11n.pt",
        "description": "Baseline liviano con YOLO11 nano.",
    },
    "training": {
        "epochs": 30,
        "imgsz": 640,
        "batch": 16,
        "patience": 10,
        "optimizer": "auto",
        "lr0": 0.01,
        "seed": 42,
    },
    "output": {
        "project": str(DRIVE_RUNS_DIR),
        "save_weights": True,
    },
}

experiment_name = experiment_config["experiment"]["name"]
model_name = experiment_config["experiment"]["model"]

print("Experimento:", experiment_name)
print("Modelo:", model_name)
print(experiment_config)

In [ ]:
# ============================================================
# Función de entrenamiento a partir de la config
# ============================================================

from ultralytics import YOLO
import pandas as pd


def train_from_config(config: dict, data_yaml: Path) -> dict:
    """
    Entrena un modelo YOLO a partir de un diccionario de configuración.
    Los resultados se guardan en una carpeta específica por experimento,
    dentro de Google Drive.
    """
    exp = config["experiment"]
    train_cfg = config["training"]
    output_cfg = config["output"]

    experiment_name = exp["name"]
    model_name = exp["model"]
    project_dir = Path(output_cfg["project"])

    project_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Experimento: {experiment_name}")
    print(f"Familia: {exp['family']}")
    print(f"Modelo: {model_name}")
    print(f"Resultados en: {project_dir / experiment_name}")
    print("=" * 80)

    start_time = time.time()

    model = YOLO(model_name)

    results = model.train(
        data=str(data_yaml),
        epochs=train_cfg["epochs"],
        imgsz=train_cfg["imgsz"],
        batch=train_cfg["batch"],
        patience=train_cfg["patience"],
        optimizer=train_cfg["optimizer"],
        lr0=train_cfg["lr0"],
        seed=train_cfg["seed"],
        project=str(project_dir),
        name=experiment_name,
        exist_ok=True,
        plots=True,
    )

    elapsed_time = time.time() - start_time

    experiment_dir = project_dir / experiment_name
    best_model_path = experiment_dir / "weights" / "best.pt"
    last_model_path = experiment_dir / "weights" / "last.pt"
    results_csv = experiment_dir / "results.csv"

    summary = {
        "experiment": experiment_name,
        "family": exp["family"],
        "model": model_name,
        "epochs": train_cfg["epochs"],
        "imgsz": train_cfg["imgsz"],
        "batch": train_cfg["batch"],
        "training_time_min": round(elapsed_time / 60, 2),
        "experiment_dir": str(experiment_dir),
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "results_csv": str(results_csv),
        "best_exists": best_model_path.exists(),
        "last_exists": last_model_path.exists(),
    }

    print("\nEntrenamiento finalizado.")
    print("Tiempo total [min]:", summary["training_time_min"])
    print("Best model:", best_model_path)
    print("Last model:", last_model_path)

    return summary

In [ ]:
# ============================================================
# Ejecución del entrenamiento
# ============================================================

RUN_TRAINING = True

if RUN_TRAINING:
    experiment_summary = train_from_config(
        config=experiment_config,
        data_yaml=DFIRE_YAML,
    )

    summary_df = pd.DataFrame([experiment_summary])
    display(summary_df)
else:
    # resume training (solo si ya existe un last.pt de una corrida previa
    # de este mismo experimento, guardado en Drive)
    model = YOLO(f"{DRIVE_RUNS_DIR}/{experiment_name}/weights/last.pt")
    results = model.train(resume=True)

In [ ]:
# ============================================================
# Verificación de pesos guardados
# ============================================================

experiment_dir = DRIVE_RUNS_DIR / experiment_name

best_model_path = experiment_dir / "weights" / "best.pt"
last_model_path = experiment_dir / "weights" / "last.pt"

print("Carpeta del experimento:", experiment_dir)
print("Best model existe:", best_model_path.exists(), best_model_path)
print("Last model existe:", last_model_path.exists(), last_model_path)

In [ ]:
# ============================================================
# Copia local (dentro de Colab) de los resultados principales
# ============================================================

import shutil

LOCAL_RESULTS_DIR = Path(f"/content/reports_{experiment_name}")
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
]

for filename in files_to_copy:
    src = experiment_dir / filename
    dst = LOCAL_RESULTS_DIR / filename

    if src.exists():
        shutil.copy(src, dst)
        print("Copiado:", dst)
    else:
        print("No encontrado:", src)

print("Carpeta lista para descargar:", LOCAL_RESULTS_DIR)
print("Uso el panel de archivos de Colab (icono de carpeta a la izquierda) para bajarla.")

In [ ]:
# ============================================================
# Validación del modelo entrenado (best.pt)
# ============================================================

model = YOLO(str(best_model_path))

metrics = model.val(data=str(DFIRE_YAML))  # mismo yaml usado en el entrenamiento
print(metrics.box.map)     # mAP50-95
print(metrics.box.map50)   # mAP50
print(metrics.box.map75)   # mAP75

In [ ]:
# ============================================================
# Inferencia sobre imágenes de test
# ============================================================

results = model.predict(
    source=str(DATA_DIR / "test" / "images"),
    save=True,
    conf=0.25,
)

# mostrar una imagen resultado en el notebook
results[0].show()

In [ ]:
# ============================================================
# Curvas de entrenamiento y matriz de confusión
# ============================================================

from PIL import Image
from IPython.display import display as ipy_display

for img_name in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png"]:
    img_path = experiment_dir / img_name
    if img_path.exists():
        print(img_name)
        ipy_display(Image.open(img_path))
    else:
        print("No encontrado:", img_path)

In [ ]:
# ============================================================
# Tabla de métricas por época
# ============================================================

df_results = pd.read_csv(experiment_dir / "results.csv")
df_results.tail()